# Meteo prognoza — XGBoost



## 1. Importi i konfiguracija

In [14]:
!pip install xgboost
%pip install optuna


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Ako nedostaju paketi, pokreni: %pip install lightgbm xgboost
import gc
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_NAME = 'XGBoost'

RANDOM_STATE = 42
HORIZONT = 12
VALIDATION_FRACTION = 0.10
BUFFER_STUPNJEVI = 0.3
CSV_PATH = 'ERA5_SPOJENO_2015_2023.csv'
OUTPUT_RESULTS = f'{MODEL_NAME.lower()}_rezultati_po_targetu.csv'

LAT_PULA, LON_PULA = 44.8666, 13.8496
LAT_RIJEKA, LON_RIJEKA = 45.3271, 14.4422

TARGETS = ['t2m_c', 'wind_speed_ms', 'swh', 'mwp', 'mwd', 'sst_c', 'msl_hpa', 'tcc', 'cape', 'blh']
LAG_VARS = ['t2m_c', 'msl_hpa', 'wind_speed_ms', 'tcc', 'cape']

# Ako nedostaje RAM-a, postavi npr. 300_000.
MAX_TRAIN_ROWS = None

## 2. Učitavanje podataka

In [4]:
df = pd.read_csv(CSV_PATH)
df['valid_time'] = pd.to_datetime(df['valid_time'])
df = df.sort_values(['latitude', 'longitude', 'valid_time']).reset_index(drop=True)

print(f'Ukupno redaka: {len(df):,}')
print(f'Broj stupaca: {df.shape[1]}')
print(f'Broj lokacija: {df[["latitude", "longitude"]].drop_duplicates().shape[0]}')
print(f'Period: {df["valid_time"].min()} do {df["valid_time"].max()}')
print('Duplikati:', df.duplicated(['valid_time', 'latitude', 'longitude']).sum())

Ukupno redaka: 3,367,224
Broj stupaca: 37
Broj lokacija: 42
Period: 2015-01-01 00:00:00 do 2023-12-31 23:00:00
Duplikati: 53928


## 3. Izdvajanje lokacija

In [6]:
locations = df[['latitude', 'longitude']].drop_duplicates().copy()
locations['dist_pula'] = np.sqrt((locations['latitude'] - LAT_PULA)**2 + (locations['longitude'] - LON_PULA)**2)
locations['dist_rijeka'] = np.sqrt((locations['latitude'] - LAT_RIJEKA)**2 + (locations['longitude'] - LON_RIJEKA)**2)
pula_grid = locations.loc[locations['dist_pula'].idxmin(), ['latitude', 'longitude']]
rijeka_grid = locations.loc[locations['dist_rijeka'].idxmin(), ['latitude', 'longitude']]
lat_pula_grid, lon_pula_grid = pula_grid['latitude'], pula_grid['longitude']
lat_rijeka_grid, lon_rijeka_grid = rijeka_grid['latitude'], rijeka_grid['longitude']

df['dist_pula'] = np.sqrt((df['latitude'] - LAT_PULA)**2 + (df['longitude'] - LON_PULA)**2)
df['dist_rijeka'] = np.sqrt((df['latitude'] - LAT_RIJEKA)**2 + (df['longitude'] - LON_RIJEKA)**2)
je_pula = (df['latitude'] == lat_pula_grid) & (df['longitude'] == lon_pula_grid)
je_rijeka = (df['latitude'] == lat_rijeka_grid) & (df['longitude'] == lon_rijeka_grid)
blizu = (df['dist_pula'] < BUFFER_STUPNJEVI) | (df['dist_rijeka'] < BUFFER_STUPNJEVI)

df_pula = df.loc[je_pula].copy()
df_rijeka = df.loc[je_rijeka].copy()
df_train = df.loc[~je_pula & ~je_rijeka & ~blizu].copy()

if MAX_TRAIN_ROWS is not None and len(df_train) > MAX_TRAIN_ROWS:
    df_train = df_train.sort_values(['valid_time', 'latitude', 'longitude']).iloc[:MAX_TRAIN_ROWS].copy()

print(f'Pula: {len(df_pula):,}')
print(f'Rijeka: {len(df_rijeka):,}')
print(f'Trening: {len(df_train):,}')
del locations
gc.collect()

Pula: 80,172
Rijeka: 80,172
Trening: 2,725,848


0

## 4. Feature engineering

In [7]:
def add_features(data):
    data = data.sort_values(['latitude', 'longitude', 'valid_time']).copy()
    groups = data.groupby(['latitude', 'longitude'])

    for var in LAG_VARS:
        if var not in data.columns:
            continue
        data[f'{var}_lag3h'] = groups[var].shift(3)
        data[f'{var}_lag6h'] = groups[var].shift(6)
        data[f'{var}_trend3h'] = data[var] - data[f'{var}_lag3h']
        data[f'{var}_rolling6h_mean'] = groups[var].transform(lambda s: s.rolling(6, min_periods=1).mean())

    if 'wind_dir_deg' in data.columns:
        data['wind_dir_sin'] = np.sin(np.radians(data['wind_dir_deg']))
        data['wind_dir_cos'] = np.cos(np.radians(data['wind_dir_deg']))

    hour = data['valid_time'].dt.hour
    day = data['valid_time'].dt.dayofyear
    data['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    data['day_sin'] = np.sin(2 * np.pi * day / 365.25)
    data['day_cos'] = np.cos(2 * np.pi * day / 365.25)

    if {'t2m_c', 'd2m_c'}.issubset(data.columns):
        data['temp_dewpoint_diff'] = data['t2m_c'] - data['d2m_c']
    if 'msl_hpa' in data.columns:
        data['pressure_gradient'] = groups['msl_hpa'].diff()
    return data

df_train = add_features(df_train)
df_pula = add_features(df_pula)
df_rijeka = add_features(df_rijeka)
print('Broj stupaca nakon feature engineeringa:', df_train.shape[1])

Broj stupaca nakon feature engineeringa: 67


## 5. Future targeti

In [8]:
def add_future_targets(data):
    data = data.sort_values(['latitude', 'longitude', 'valid_time']).copy()
    groups = data.groupby(['latitude', 'longitude'])
    for target in TARGETS:
        if target in data.columns:
            data[f'{target}_future'] = groups[target].shift(-HORIZONT)
    return data

df_train = add_future_targets(df_train)
df_pula = add_future_targets(df_pula)
df_rijeka = add_future_targets(df_rijeka)
for target in TARGETS:
    col = f'{target}_future'
    if col in df_train.columns:
        print(f'{target}: {df_train[col].notna().sum():,} valjanih targeta')

t2m_c: 2,681,784 valjanih targeta
wind_speed_ms: 2,681,784 valjanih targeta
swh: 631,008 valjanih targeta
mwp: 631,008 valjanih targeta
mwd: 631,008 valjanih targeta
sst_c: 1,340,892 valjanih targeta
msl_hpa: 2,681,784 valjanih targeta
tcc: 2,681,784 valjanih targeta
cape: 2,681,784 valjanih targeta
blh: 2,681,784 valjanih targeta


## 6. Priprema featurea i model

In [9]:
DROP_COLUMNS = ['valid_time', 'latitude', 'longitude', 'dist_pula', 'dist_rijeka', 'hour', 'day']

def prepare_X(data, feature_names=None, medians=None):
    drop_cols = [c for c in DROP_COLUMNS if c in data.columns]
    drop_cols += [c for c in data.columns if c.endswith('_future')]
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    selected_cols = [c for c in numeric_cols if c not in drop_cols]

    if feature_names is not None:
        selected_cols = [c for c in feature_names if c in data.columns]

    X = data.loc[:, selected_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    if medians is None:
        medians = X.median()
    X = X.fillna(medians)

    if feature_names is not None:
        X = X.reindex(columns=feature_names, fill_value=0)

    X = X.astype(np.float32)
    medians = medians.astype(np.float32)
    return X, medians

def make_model(name):
    if name == 'LightGBM':
        return lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=10,
            num_leaves=31, min_child_samples=20,
            reg_alpha=0.5, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=2, verbosity=-1
        )

    if name == 'XGBoost':
        return xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=8,
            min_child_weight=3, subsample=0.85, colsample_bytree=0.9,
            reg_alpha=0.5, reg_lambda=1.0,
            objective='reg:squarederror', random_state=RANDOM_STATE,
            n_jobs=2, tree_method='hist'
        )

    raise ValueError('MODEL_NAME mora biti LightGBM ili XGBoost')

## 7. Trening jednog targeta

In [10]:
def train_one_target(model_name, target, train_data, test_dict):
    target_col = f'{target}_future'
    if target_col not in train_data.columns:
        print(f'{target}: target ne postoji')
        return None

    valid_idx = train_data[target_col].notna()
    n_valid = int(valid_idx.sum())
    if n_valid < 1000:
        print(f'{target}: preskačem, samo {n_valid} valjanih redaka')
        return None

    target_data = train_data.loc[valid_idx]
    y = target_data[target_col].astype(np.float32)
    X, medians = prepare_X(target_data)
    feature_names = X.columns.tolist()

    cutoff = int(len(X) * (1 - VALIDATION_FRACTION))
    X_fit, X_val = X.iloc[:cutoff], X.iloc[cutoff:]
    y_fit, y_val = y.iloc[:cutoff], y.iloc[cutoff:]

    print(f'{MODEL_NAME}/{target}: treniranje na {len(X_fit):,} redaka i {len(feature_names)} featurea')
    model = make_model(model_name)
    model.fit(X_fit, y_fit)
    pred_val = model.predict(X_val)

    result = {
        'model': model_name,
        'target': target,
        'horizon_hours': HORIZONT,
        'MAE_val': mean_absolute_error(y_val, pred_val),
        'RMSE_val': np.sqrt(mean_squared_error(y_val, pred_val)),
        'R2_val': r2_score(y_val, pred_val),
        'n_train': len(X_fit),
        'n_validation': len(X_val),
        'n_features': len(feature_names)
    }

    for location, test_data in test_dict.items():
        test_valid_idx = test_data[target_col].notna()
        if test_valid_idx.sum() < 10:
            result[f'MAE_{location}'] = np.nan
            result[f'RMSE_{location}'] = np.nan
            result[f'R2_{location}'] = np.nan
            continue

        test_subset = test_data.loc[test_valid_idx]
        X_test, _ = prepare_X(test_subset, feature_names, medians)
        y_test = test_subset[target_col].astype(np.float32)
        pred_test = model.predict(X_test)
        result[f'MAE_{location}'] = mean_absolute_error(y_test, pred_test)
        result[f'RMSE_{location}'] = np.sqrt(mean_squared_error(y_test, pred_test))
        result[f'R2_{location}'] = r2_score(y_test, pred_test)

    print(f"{target}: MAE={result['MAE_val']:.3f}, RMSE={result['RMSE_val']:.3f}, R2={result['R2_val']:.3f}")
    del target_data, X, X_fit, X_val, y, y_fit, y_val, pred_val, model
    if 'X_test' in locals():
        del X_test
    if 'y_test' in locals():
        del y_test
    if 'pred_test' in locals():
        del pred_test
    gc.collect()
    return result

## 8. Pokretanje odabranog modela

In [10]:
if MODEL_NAME not in ['LightGBM', 'XGBoost']:
    raise ValueError('MODEL_NAME mora biti LightGBM ili XGBoost')

test_dict = {'pula': df_pula, 'rijeka': df_rijeka}
results = []

for target in TARGETS:
    result = train_one_target(MODEL_NAME, target, df_train, test_dict)
    if result is not None:
        results.append(result)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_RESULTS, index=False)
print('\n=== REZULTATI ===')
display(results_df)
print(f'Spremljeno u: {OUTPUT_RESULTS}')

XGBoost/t2m_c: treniranje na 2,413,605 redaka i 62 featurea
t2m_c: MAE=1.187, RMSE=1.567, R2=0.965
XGBoost/wind_speed_ms: treniranje na 2,413,605 redaka i 62 featurea
wind_speed_ms: MAE=0.700, RMSE=0.896, R2=0.489
XGBoost/swh: treniranje na 567,907 redaka i 62 featurea
swh: MAE=0.107, RMSE=0.160, R2=0.737
XGBoost/mwp: treniranje na 567,907 redaka i 62 featurea
mwp: MAE=0.249, RMSE=0.326, R2=0.839
XGBoost/mwd: treniranje na 567,907 redaka i 62 featurea
mwd: MAE=42.258, RMSE=65.155, R2=0.360
XGBoost/sst_c: treniranje na 1,206,802 redaka i 62 featurea
sst_c: MAE=0.149, RMSE=0.257, R2=0.998
XGBoost/msl_hpa: treniranje na 2,413,605 redaka i 62 featurea
msl_hpa: MAE=1.304, RMSE=1.737, R2=0.951
XGBoost/tcc: treniranje na 2,413,605 redaka i 62 featurea
tcc: MAE=0.191, RMSE=0.240, R2=0.597
XGBoost/cape: treniranje na 2,413,605 redaka i 62 featurea
cape: MAE=84.364, RMSE=208.923, R2=0.695
XGBoost/blh: treniranje na 2,413,605 redaka i 62 featurea
blh: MAE=157.420, RMSE=226.985, R2=0.777

=== REZU

,model,target,horizon_hours,MAE_val,RMSE_val,R2_val,n_train,n_validation,n_features,MAE_pula,RMSE_pula,R2_pula,MAE_rijeka,RMSE_rijeka,R2_rijeka
0,XGBoost,t2m_c,12,1.187027,1.567361,0.964987,2413605,268179,62,0.535815,0.712913,0.987275,0.913477,1.195605,0.975603
1,XGBoost,wind_speed_ms,12,0.699967,0.896214,0.488842,2413605,268179,62,1.257135,1.643799,0.689290,0.745837,0.980964,0.653582
2,XGBoost,swh,12,0.106542,0.160039,0.737064,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
3,XGBoost,mwp,12,0.248681,0.325990,0.839041,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
4,XGBoost,mwd,12,42.257633,65.154867,0.360047,567907,63101,62,NaN,NaN,NaN,NaN,NaN,NaN
5,XGBoost,sst_c,12,0.148785,0.257010,0.998153,1206802,134090,62,0.080476,0.129849,0.999410,NaN,NaN,NaN
6,XGBoost,msl_hpa,12,1.303655,1.736569,0.951220,2413605,268179,62,0.937559,1.249137,0.973440,0.989978,1.311496,0.969918
7,XGBoost,tcc,12,0.190615,0.240316,0.597187,2413605,268179,62,0.183649,0.228146,0.651888,0.167465,0.212526,0.686012
8,XGBoost,cape,12,84.364494,208.922758,0.694732,2413605,268179,62,120.421532,261.137251,0.863477,77.541100,172.729750,0.747434
9,XGBoost,blh,12,157.420212,226.984787,0.776803,2413605,268179,62,117.658325,163.728959,0.765338,135.917328,194.550353,0.789077


Spremljeno u: xgboost_rezultati_po_targetu.csv


## 9. Optune
+12 sati



In [19]:
import gc
import optuna
import xgboost as xgb

from sklearn.metrics import mean_absolute_error

OPTUNA_TRIALS = 20
OPTUNA_MAX_ROWS = None
XGBOOST_THREADS = 4

In [20]:
def objective_xgboost(trial, target):
    target_col = f'{target}_future'

    valid_idx = df_train[target_col].notna()
    data = df_train.loc[valid_idx]

    if (
        OPTUNA_MAX_ROWS is not None
        and len(data) > OPTUNA_MAX_ROWS
    ):
        data = (
            data
            .sort_values(
                ['valid_time', 'latitude', 'longitude']
            )
            .iloc[:OPTUNA_MAX_ROWS]
        )

    y = data[target_col].astype(np.float32)
    X, _ = prepare_X(data)

    cutoff = int(len(X) * 0.90)

    X_fit = X.iloc[:cutoff]
    X_val = X.iloc[cutoff:]
    y_fit = y.iloc[:cutoff]
    y_val = y.iloc[cutoff:]

    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'n_jobs': XGBOOST_THREADS,

        'n_estimators': trial.suggest_int(
            'n_estimators',
            200,
            800
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            3,
            12
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.15,
            log=True
        ),

        'min_child_weight': trial.suggest_int(
            'min_child_weight',
            1,
            20
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.6,
            1.0
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            0.6,
            1.0
        ),

        'gamma': trial.suggest_float(
            'gamma',
            0.0,
            5.0
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha',
            1e-8,
            10.0,
            log=True
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda',
            1e-8,
            10.0,
            log=True
        )
    }

    model = xgb.XGBRegressor(**params)

    model.fit(
        X_fit,
        y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    pred = model.predict(X_val)

    score = mean_absolute_error(
        y_val,
        pred
    )

    del (
        data,
        X,
        X_fit,
        X_val,
        y,
        y_fit,
        y_val,
        model,
        pred
    )

    gc.collect()

    return score

In [21]:
optuna_results = []
studies_xgb = {}

for target in TARGETS:
    target_col = f'{target}_future'

    if target_col not in df_train.columns:
        print(
            f'{target}: target ne postoji, preskačem.'
        )
        continue

    valid_count = int(
        df_train[target_col].notna().sum()
    )

    if valid_count < 1000:
        print(
            f'{target}: premalo podataka '
            f'({valid_count}), preskačem.'
        )
        continue

    print(
        f'\n===== OPTUNA XGBOOST: {target} ====='
    )

    study = optuna.create_study(
        direction='minimize',
        study_name=f'xgboost_{target}',
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        ),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=2,
            n_warmup_steps=10
        )
    )

    study.optimize(
        lambda trial, current_target=target:
            objective_xgboost(
                trial,
                current_target
            ),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=True
    )

    studies_xgb[target] = study

    optuna_results.append({
        'model': 'XGBoost',
        'target': target,
        'best_MAE_optuna': study.best_value,
        'n_trials': len(study.trials),
        **study.best_params
    })

    print(
        f'Najbolji MAE za {target}: '
        f'{study.best_value:.4f}'
    )

    print(
        'Najbolji parametri:',
        study.best_params
    )

optuna_xgb_df = pd.DataFrame(
    optuna_results
)

optuna_xgb_df.to_csv(
    'xgboost_optuna_svi_targeti.csv',
    index=False
)

print('\n=== XGBOOST OPTUNA REZULTATI ===')
display(optuna_xgb_df)

print(
    '\nSpremljeno u: '
    'xgboost_optuna_svi_targeti.csv'
)

[I 2026-08-22 00:02:52,427] A new study created in memory with name: xgboost_t2m_c



===== OPTUNA XGBOOST: t2m_c =====


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-22 00:06:51,550] Trial 0 finished with value: 0.829261302947998 and parameters: {'n_estimators': 425, 'max_depth': 12, 'learning_rate': 0.07259248719561363, 'min_child_weight': 12, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.829261302947998.
[I 2026-08-22 00:09:11,176] Trial 1 finished with value: 1.548059105873108 and parameters: {'n_estimators': 625, 'max_depth': 3, 'learning_rate': 0.13826189316223852, 'min_child_weight': 17, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 0 with value: 0.829261302947998.
[I 2026-08-22 00:11:15,066] Trial 2 finished with value: 1.4729101657867432 and parameters: {'n_estimators': 459, 'max_depth': 5, 'learning_rate': 0.05243180891902853, 'min_child

[I 2026-08-22 01:03:46,150] A new study created in memory with name: xgboost_wind_speed_ms


...

[I 2026-08-22 06:08:07,928] Trial 19 finished with value: 68.4115982055664 and parameters: {'n_estimators': 796, 'max_depth': 10, 'learning_rate': 0.06193229498222443, 'min_child_weight': 1, 'subsample': 0.879713599128111, 'colsample_bytree': 0.6693761544549911, 'gamma': 3.1631519638115915, 'reg_alpha': 2.726680493037923e-05, 'reg_lambda': 2.347372757484223e-05}. Best is trial 18 with value: 64.22740173339844.
Najbolji MAE za cape: 64.2274
Najbolji parametri: {'n_estimators': 725, 'max_depth': 11, 'learning_rate': 0.10796100656646772, 'min_child_weight': 1, 'subsample': 0.8505919001558102, 'colsample_bytree': 0.7613495462127717, 'gamma': 2.819873495107165, 'reg_alpha': 0.00014021547203370562, 'reg_lambda': 3.9233028601151885e-05}

===== OPTUNA XGBOOST: blh =====


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-22 06:12:02,734] Trial 0 finished with value: 119.74598693847656 and parameters: {'n_estimators': 425, 'max_depth': 12, 'learning_rate': 0.07259248719561363, 'min_child_weight': 12, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 119.74598693847656.
[I 2026-08-22 06:14:18,147] Trial 1 finished with value: 196.59645080566406 and parameters: {'n_estimators': 625, 'max_depth': 3, 'learning_rate': 0.13826189316223852, 'min_child_weight': 17, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 0 with value: 119.74598693847656.
[I 2026-08-22 06:16:19,637] Trial 2 finished with value: 187.75982666015625 and parameters: {'n_estimators': 459, 'max_depth': 5, 'learning_rate': 0.05243180891902853, 'min_c

,model,target,best_MAE_optuna,n_trials,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,XGBoost,t2m_c,0.809330,20,725,11,0.088477,5,0.639080,0.611988,1.328913,5.365164e-05,0.000033
1,XGBoost,wind_speed_ms,0.546984,20,725,11,0.107961,1,0.850592,0.761350,2.819873,1.402155e-04,0.000039
2,XGBoost,swh,0.101628,20,798,10,0.024196,20,0.813253,0.850165,0.055191,1.074018e-03,0.002114
3,XGBoost,mwp,0.231638,20,682,10,0.092385,8,0.798282,0.731282,0.117509,1.593685e-02,0.022098
4,XGBoost,mwd,40.807728,20,658,11,0.035306,3,0.866105,0.858713,3.750695,3.359184e-07,0.000069
5,XGBoost,sst_c,0.147311,20,799,9,0.026230,20,0.814250,0.998611,3.766898,1.628979e-03,9.768192
6,XGBoost,msl_hpa,0.803172,20,425,12,0.072592,12,0.662407,0.662398,0.290418,6.245760e-01,0.002571
7,XGBoost,tcc,0.151077,20,425,12,0.072592,12,0.662407,0.662398,0.290418,6.245760e-01,0.002571
8,XGBoost,cape,64.227402,20,725,11,0.107961,1,0.850592,0.761350,2.819873,1.402155e-04,0.000039
9,XGBoost,blh,119.745987,20,425,12,0.072592,12,0.662407,0.662398,0.290418,6.245760e-01,0.002571



Spremljeno u: xgboost_optuna_svi_targeti.csv


In [ ]:
import gc
import optuna
import xgboost as xgb

from sklearn.metrics import mean_absolute_error

OPTUNA_TRIALS = 15
OPTUNA_MAX_ROWS = None
XGBOOST_THREADS = 4